Add geometry to Dragna et al: in total 964 / 1489 are matched (12/2025)

In [2]:
import pandas as pd
import geopandas as gpd
from land_cover.load import loadDranga17, dranga_shoreline_pth, load_prelim_matchup_data, load_gee_input, load_lit_review_lakes_jn_pld, load_gee_input, load_gee_digitized
from land_cover.utils import create_unique_index

%load_ext autoreload
%autoreload 2

In [ ]:
"""Loads all lakes and merges in PLD matches via double join
merges in digitized geoms via additional join
merges in dragna DOC data via additional join
geometry has duplicates because some lakes were sampled at the exact same reported lat/Long
"""

"Loads all lakes and merges in PLD matches via double join\nmerges in digitized geoms via additional join\nmerges in dragna DOC data via additional join\nI'll put his duplicates because some lakes were sampled at the exact same reported lat/Long\n"

In [4]:
# has lat/lon, gives us TopoCat lake_id for next join and has sample no which can link to DOC value!
df_matchup_data = load_prelim_matchup_data()
df_matchup_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1489 entries, 1692 to 8713
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Name       1489 non-null   object 
 1   lake_id    395 non-null    float64
 2   Sample No  1489 non-null   float64
 3   Latitude   1489 non-null   float64
 4   Longitude  1489 non-null   float64
dtypes: float64(4), object(1)
memory usage: 69.8+ KB


In [5]:
# has SampleUID and Lat/Lon; includes some lit review not from Dranga
df_gee_input = load_gee_input(
    source="LitReviewLakes"
)  # .sort_values(["Latitude", "Longitude"])
df_gee_input.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1577 entries, 6114 to 8713
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CurrentlyM  1577 non-null   int64  
 1   Latitude    1577 non-null   float64
 2   Longitude   1577 non-null   float64
 3   SampleUID   1577 non-null   object 
 4   Source      1577 non-null   object 
 5   WesternHem  1577 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(2)
memory usage: 75.5+ KB


In [6]:
# First join to get SampleUID, which can join in GEE digitizing
df = df_matchup_data.merge(
    df_gee_input, on=["Latitude", "Longitude"], how="left"
)  # has dups because some samples have same lat/lon

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1897 entries, 0 to 1896
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Name        1897 non-null   object 
 1   lake_id     462 non-null    float64
 2   Sample No   1897 non-null   float64
 3   Latitude    1897 non-null   float64
 4   Longitude   1897 non-null   float64
 5   CurrentlyM  1897 non-null   int64  
 6   SampleUID   1897 non-null   object 
 7   Source      1897 non-null   object 
 8   WesternHem  1897 non-null   bool   
dtypes: bool(1), float64(4), int64(1), object(3)
memory usage: 120.5+ KB


In [7]:
len(df.SampleUID.unique())

1513

In [8]:
# Example duplicats that can be merged, collapsing SampleUID
df.iloc[100:104, :]

,Name,lake_id,Sample No,Latitude,Longitude,CurrentlyM,SampleUID,Source,WesternHem
100,LitReviewLakes,8.221991e+09,582.0,65.18,-112.33,1,LRL_00000584,LitReviewLakes,True
101,LitReviewLakes,8.222984e+09,467.0,63.16,-112.28,1,LRL_00000469,LitReviewLakes,True
102,LitReviewLakes,8.222984e+09,467.0,63.16,-112.28,1,LRL_00001502,LitReviewLakes,True
103,LitReviewLakes,8.223005e+09,528.0,63.25,-111.43,1,LRL_00000530,LitReviewLakes,True


In [9]:
df = df.groupby("Sample No").first().reset_index()  # hot fix to remove dups
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1489 entries, 0 to 1488
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Sample No   1489 non-null   float64
 1   Name        1489 non-null   object 
 2   lake_id     395 non-null    float64
 3   Latitude    1489 non-null   float64
 4   Longitude   1489 non-null   float64
 5   CurrentlyM  1489 non-null   int64  
 6   SampleUID   1489 non-null   object 
 7   Source      1489 non-null   object 
 8   WesternHem  1489 non-null   bool   
dtypes: bool(1), float64(4), int64(1), object(3)
memory usage: 94.6+ KB


In [10]:
# Has PLD geometries and lake_id
gdf_jn_PLD = load_lit_review_lakes_jn_pld()
gdf_jn_PLD.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   lake_id   391 non-null    float64 
 1   geometry  391 non-null    geometry
dtypes: float64(1), geometry(1)
memory usage: 6.2 KB


In [11]:
# second join to get geometries from PLD. NOTE: right index "lake_id" (From TopoCat) has dups bc some lakes sampled multiple times
gdf = gdf_jn_PLD.merge(df, on="lake_id", how="right")
gdf.info()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1489 entries, 0 to 1488
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lake_id     395 non-null    float64 
 1   geometry    395 non-null    geometry
 2   Sample No   1489 non-null   float64 
 3   Name        1489 non-null   object  
 4   Latitude    1489 non-null   float64 
 5   Longitude   1489 non-null   float64 
 6   CurrentlyM  1489 non-null   int64   
 7   SampleUID   1489 non-null   object  
 8   Source      1489 non-null   object  
 9   WesternHem  1489 non-null   bool    
dtypes: bool(1), float64(4), geometry(1), int64(1), object(3)
memory usage: 106.3+ KB


In [12]:
df_digitzed = load_gee_digitized()
df_digitzed.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 561 entries, 229 to 583
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   PointLat   561 non-null    object  
 1   PointLon   561 non-null    object  
 2   caption    561 non-null    object  
 3   pld_match  561 non-null    object  
 4   SampleUID  561 non-null    object  
 5   savetype   561 non-null    object  
 6   geometry   561 non-null    geometry
dtypes: geometry(1), object(6)
memory usage: 35.1+ KB


In [13]:
# Join in based on sample_uid
gdf_out = gdf.merge(
    df_digitzed,  # .drop(columns=["geometry"]),
    on="SampleUID",
    how="left",
)
# use geometry_x (PLD) unless geometry_y (digitized) is not null
gdf_out["geometry"] = gdf_out["geometry_y"].combine_first(gdf_out["geometry_x"])
gdf_out.drop(columns=["geometry_x", "geometry_y"], inplace=True)
gdf_out.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1489 entries, 0 to 1488
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lake_id     395 non-null    float64 
 1   Sample No   1489 non-null   float64 
 2   Name        1489 non-null   object  
 3   Latitude    1489 non-null   float64 
 4   Longitude   1489 non-null   float64 
 5   CurrentlyM  1489 non-null   int64   
 6   SampleUID   1489 non-null   object  
 7   Source      1489 non-null   object  
 8   WesternHem  1489 non-null   bool    
 9   PointLat    569 non-null    object  
 10  PointLon    569 non-null    object  
 11  caption     569 non-null    object  
 12  pld_match   569 non-null    object  
 13  savetype    569 non-null    object  
 14  geometry    964 non-null    geometry
dtypes: bool(1), float64(4), geometry(1), int64(1), object(8)
memory usage: 164.4+ KB


In [15]:
gdf_out.drop(columns=["WesternHem", "PointLat", "PointLon", "Source", "Name"], inplace=True)

In [16]:
# Write out
gdf_out.to_file(dranga_shoreline_pth)
print(f"Wrote: {dranga_shoreline_pth}")

Wrote: /Volumes/metis/Datasets/Dranga-2017/edk_out/shp/dranga17_shorelines.gpkg


In [29]:
# join in chem
df_dranga, _, _ = loadDranga17()
gdf_chem = gdf_out.merge(
    df_dranga.drop(columns=["Latitude", "Longitude"]), on="Sample No", validate="1:1"
)
gdf_chem.info(max_cols=500)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1489 entries, 0 to 1488
Data columns (total 48 columns):
 #   Column                                            Non-Null Count  Dtype   
---  ------                                            --------------  -----   
 0   lake_id                                           395 non-null    float64 
 1   Sample No                                         1489 non-null   float64 
 2   Latitude                                          1489 non-null   float64 
 3   Longitude                                         1489 non-null   float64 
 4   CurrentlyM                                        1489 non-null   int64   
 5   SampleUID                                         1489 non-null   object  
 6   caption                                           569 non-null    object  
 7   pld_match                                         569 non-null    object  
 8   savetype                                          569 non-null    object  
 9   

In [30]:
# Write out
dranga_shoreline_chem_pth = dranga_shoreline_pth.replace(".gpkg", "_chem.gpkg")
gdf_chem.to_file(dranga_shoreline_chem_pth)
print(f"Wrote: {dranga_shoreline_chem_pth}")

Wrote: /Volumes/metis/Datasets/Dranga-2017/edk_out/shp/dranga17_shorelines_chem.gpkg
